# Lateral Load Pushover Analysis — 2D RC Frame from CADThis notebook demonstrates a **complete workflow** from CAD geometry to nonlinearpushover analysis using:- **Gmsh** (via IGES import) for frame discretisation- **pyGmsh** for mesh control and OpenSees bridging- **`RectangularColumnSection`** for fiber section generation- **OpenSeesPy** for nonlinear static (pushover) analysis## Frame Geometry (from `acad/frame_2D.iges`)| Property | Value ||---|---|| Bays | 3 × 4 000 mm || Stories | 2 × 3 000 mm || Columns (exterior) | C80×80 — at x = 0 and x = 8 000 mm || Columns (interior) | C40×40 — at x = 4 000 and x = 12 000 mm || Beams | V30×50 — all levels || Supports | Fixed at base (x = 0, 8 000, 12 000 mm) || Lateral loads | Applied at left column, 1st and 2nd floor |

In [ ]:
import sys, osimport numpy as npimport matplotlib.pyplot as pltsys.path.insert(0, '../src')import gmshfrom apeGmsh import apeGmshimport openseespy.opensees as opsfrom sections import RectangularColumnSectionprint("All imports OK")

---## Part 1 — Define Fiber SectionsEach member type gets a Gmsh-meshed fiber section via `RectangularColumnSection`.These are created first because the class runs its own internal Gmsh session.| Member | b × h [mm] | Bars (top/bot) | Side bars | f'c [MPa] ||---|---|---|---|---|| C80×80 | 800 × 800 | 4 ø 25 | 3 ø 20 | 30 || C40×40 | 400 × 400 | 3 ø 20 | 2 ø 16 | 30 || V30×50 | 300 × 500 | 3 ø 20 | 2 ø 12 | 30 |

In [ ]:
fc  = 30.0;  fy = 420.0;  Es = 200_000;  cf = 1.3sec_C80 = RectangularColumnSection(    b=800, h=800, cover=40,    top_bars=(4, 25), bot_bars=(4, 25), side_bars=(3, 20),    fc=fc, fy=fy, Es=Es, confinement_factor=cf, mesh_size=40,)sec_C40 = RectangularColumnSection(    b=400, h=400, cover=40,    top_bars=(3, 20), bot_bars=(3, 20), side_bars=(2, 16),    fc=fc, fy=fy, Es=Es, confinement_factor=cf, mesh_size=25,)sec_V30 = RectangularColumnSection(    b=300, h=500, cover=30,    top_bars=(3, 20), bot_bars=(3, 20), side_bars=(2, 12),    fc=fc, fy=fy, Es=Es, confinement_factor=cf, mesh_size=25,)for name, sec in [("C80x80", sec_C80), ("C40x40", sec_C40), ("V30x50", sec_V30)]:    f = sec.get_fibers()    print(f"{name}: {f.n_fibers} fibers, Ag = {sec.Ag:.0f} mm², rho = {sec.rho*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))for ax, sec, title in zip(axes, [sec_C80, sec_C40, sec_V30], ['C80×80', 'C40×40', 'V30×50']):    sec.plot(ax=ax, show=False)    ax.set_title(title)plt.tight_layout()plt.show()

---## Part 2 — Import CAD Geometry and Mesh the FrameLoad the IGES file, merge duplicates, assign physical groups, andgenerate a transfinite 1D mesh (4 elements per member).

In [ ]:
g = apeGmsh(model_name="Frame_2D_Pushover", verbose=True)g.begin()# ── Load IGES from AutoCAD ──gmsh.model.occ.importShapes("../acad/frame_2D.iges")gmsh.model.occ.removeAllDuplicates()  # merge coincident points, split overlapping curvesgmsh.model.occ.synchronize()# ── Physical groups (tags match the frame_2D.geo reference) ──g.physical.add(1, [4, 6, 10],            name="C40x40")g.physical.add(1, [5, 7, 11, 12],       name="C80x80")g.physical.add(1, [1, 2, 3, 8, 13, 14], name="V30x50")g.physical.add(0, [9, 10, 11],           name="supports")g.physical.add(0, [1],                   name="load_1")g.physical.add(0, [7],                   name="load_2")# ── Transfinite mesh ──N_ELEM = 4for tag in [4, 6, 10, 5, 7, 11, 12, 1, 2, 3, 8, 13, 14]:    g.mesh.structured.set_transfinite_curve(tag, N_ELEM + 1)g.mesh.generation.generate(1)node_tags, _, _ = gmsh.model.mesh.getNodes()print(f"\nMesh: {len(node_tags)} nodes, {N_ELEM * 13} line elements")

In [ ]:
# ── Visualise the frame ──fig, ax = plt.subplots(figsize=(12, 7))colors = {"C40x40": "green", "C80x80": "blue", "V30x50": "red"}for pg_name, color in colors.items():    for d, pg_tag in gmsh.model.getPhysicalGroups(1):        if gmsh.model.getPhysicalName(d, pg_tag) != pg_name:            continue        for ent_tag in gmsh.model.getEntitiesForPhysicalGroup(d, pg_tag):            _, _, enodes = gmsh.model.mesh.getElements(1, int(ent_tag))            for en in enodes:                for n1, n2 in en.reshape(-1, 2).astype(int):                    c1 = gmsh.model.mesh.getNode(int(n1))[0]                    c2 = gmsh.model.mesh.getNode(int(n2))[0]                    ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color=color, lw=2.5)for pt in [9, 10, 11]:    bb = gmsh.model.getBoundingBox(0, pt)    ax.plot(bb[0], bb[1], 'k^', markersize=14, zorder=5)for pt, label in [(1, 'F1'), (7, 'F2')]:    bb = gmsh.model.getBoundingBox(0, pt)    ax.annotate('', xy=(bb[0]-400, bb[1]), xytext=(bb[0]-1500, bb[1]),                arrowprops=dict(arrowstyle='->', color='orange', lw=2.5))    ax.text(bb[0]-1600, bb[1]+100, label, fontsize=11, color='orange', ha='right', fontweight='bold')from matplotlib.lines import Line2Dax.legend(handles=[    Line2D([0],[0], color='blue', lw=2.5, label='C80×80'),    Line2D([0],[0], color='green', lw=2.5, label='C40×40'),    Line2D([0],[0], color='red', lw=2.5, label='V30×50'),    Line2D([0],[0], marker='^', color='k', lw=0, ms=10, label='Supports'),], loc='upper right', fontsize=10)ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')ax.set_title('2D RC Frame — Imported from CAD (IGES)')ax.set_aspect('equal'); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

---## Part 3 — Build the OpenSees ModelUse `g.opensees` to translate the Gmsh mesh into OpenSeesPy commands.We use `elasticBeamColumn` with gross section properties and P-Deltageometric transformation for columns.

In [ ]:
# ── Gross section properties ──A_C80 = 800*800;  I_C80 = 800*800**3/12A_C40 = 400*400;  I_C40 = 400*400**3/12A_V30 = 300*500;  I_V30 = 300*500**3/12Ec = 4700 * np.sqrt(fc)# ── OpenSees bridge — declare everything except loads ──(g.opensees    .set_model(ndm=2, ndf=3)    .add_geom_transf("ColTransf",  "PDelta")    .add_geom_transf("BeamTransf", "Linear")    .assign_element("C80x80", "elasticBeamColumn",                    geom_transf="ColTransf", A=A_C80, E=Ec, Iz=I_C80)    .assign_element("C40x40", "elasticBeamColumn",                    geom_transf="ColTransf", A=A_C40, E=Ec, Iz=I_C40)    .assign_element("V30x50", "elasticBeamColumn",                    geom_transf="BeamTransf", A=A_V30, E=Ec, Iz=I_V30)    .fix("supports", dofs=[1, 1, 1]))g.opensees.build()node_df = g.opensees.node_table()elem_df = g.opensees.element_table()print(f"Model: {len(node_df)} nodes, {len(elem_df)} elements")print(f"Ec = {Ec:.0f} MPa")print(elem_df.groupby('pg_name').size().to_string())# Export (without load patterns) for referenceg.opensees.export_py("frame_model.py")print("\nExported structural model → frame_model.py")

---## Part 4 — Pushover Analysis1. Load the exported model (nodes, elements, BCs — no loads)2. Apply gravity as lumped nodal forces3. Run displacement-controlled pushover at the roof node

In [ ]:
# ── Identify control nodes from the node table ──
roof_mask   = (np.abs(node_df['x']) < 1.0) & (np.abs(node_df['y'] - 6000) < 1.0)
roof_node   = int(node_df[roof_mask].index[0])
floor1_mask = (np.abs(node_df['x']) < 1.0) & (np.abs(node_df['y'] - 3000) < 1.0)
floor1_node = int(node_df[floor1_mask].index[0])
print(f"Roof node (control): {roof_node}  at y = 6000 mm")
print(f"1st floor node:      {floor1_node}  at y = 3000 mm")

# ── Load the structural model ──
ops.wipe()
exec(open("frame_model.py").read())

# ── Gravity (self-weight + dead load) ──
W_beam = 3 * 4000 * 24e-6 * A_V30   # beam self-weight per floor [N]
W_slab = 2e-3 * 4000 * 4000          # slab dead load per floor [N] (2 kPa, 4×4m trib)
W_floor = W_beam + W_slab
W_per_node = W_floor / 4             # 4 joints per floor
print(f"Gravity per floor: {W_floor/1000:.1f} kN → {W_per_node/1000:.1f} kN per node")

ops.timeSeries('Constant', 1)
ops.pattern('Plain', 100, 1)
for nid, row in node_df.iterrows():
    if row['y'] > 100:
        ops.load(int(nid), 0.0, -W_per_node, 0.0)

ops.constraints('Plain')
ops.numberer('RCM')
ops.system('BandGeneral')
ops.test('NormDispIncr', 1.0e-8, 10)
ops.algorithm('Newton')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')
ops.analyze(1)
ops.loadConst('-time', 0.0)
print("Gravity analysis complete")

In [ ]:
# ── Pushover: displacement-controlled at roof ──
max_drift = 0.03          # 3% roof drift
max_disp  = max_drift * 6000
n_steps   = 200
d_disp    = max_disp / n_steps

ops.timeSeries('Linear', 2)
ops.pattern('Plain', 200, 2)
ops.load(roof_node, 1.0, 0.0, 0.0)

ops.test('NormDispIncr', 1.0e-8, 10)
ops.integrator('DisplacementControl', roof_node, 1, d_disp)
ops.analysis('Static')

disp_roof  = [0.0]
disp_floor = [0.0]
base_shear = [0.0]
support_nodes = g.opensees._nodes_for_pg("supports")

for i in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        ops.algorithm('NewtonLineSearch', 0.8)
        ok = ops.analyze(1)
        ops.algorithm('Newton')
    if ok != 0:
        print(f"  Pushover failed at step {i+1}/{n_steps}")
        break
    
    disp_roof.append(ops.nodeDisp(roof_node, 1))
    disp_floor.append(ops.nodeDisp(floor1_node, 1))
    ops.reactions()
    V = sum(ops.nodeReaction(nid, 1) for nid in support_nodes)
    base_shear.append(-V)

disp_roof  = np.array(disp_roof)
disp_floor = np.array(disp_floor)
base_shear = np.array(base_shear)

print(f"Pushover: {len(disp_roof)-1}/{n_steps} steps")
print(f"Max roof disp:  {disp_roof[-1]:.1f} mm  ({disp_roof[-1]/6000*100:.2f}% drift)")
print(f"Max base shear: {base_shear.max()/1000:.1f} kN")
ops.wipe()

---## Part 5 — Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# ── Pushover curve ──ax = axes[0]ax.plot(disp_roof, base_shear / 1000, 'b-', linewidth=2)ax.set_xlabel('Roof Displacement [mm]')ax.set_ylabel('Base Shear [kN]')ax.set_title('Pushover Curve')ax.grid(True, alpha=0.3)ax.set_xlim(left=0); ax.set_ylim(bottom=0)V_max_idx = np.argmax(base_shear)ax.plot(disp_roof[V_max_idx], base_shear[V_max_idx]/1000, 'ro', ms=10,        label=f'Peak: {base_shear[V_max_idx]/1000:.0f} kN @ {disp_roof[V_max_idx]:.0f} mm')ax.legend(fontsize=10)# ── Drift profile ──ax2 = axes[1]d1 = disp_floor[V_max_idx]; d2 = disp_roof[V_max_idx]drift_s1 = d1/3000*100;  drift_s2 = (d2-d1)/3000*100ax2.barh([1, 2], [drift_s1, drift_s2], color=['steelblue', 'coral'], height=0.5)ax2.set_xlabel('Inter-story Drift [%]')ax2.set_ylabel('Story')ax2.set_title(f'Drift at Peak V ({disp_roof[V_max_idx]:.0f} mm)')ax2.set_yticks([1, 2])ax2.set_yticklabels(['Story 1\n(0-3m)', 'Story 2\n(3-6m)'])ax2.grid(True, alpha=0.3, axis='x')for i, d in enumerate([drift_s1, drift_s2]):    ax2.text(d+0.02, i+1, f'{d:.2f}%', va='center', fontsize=11)plt.tight_layout(); plt.show()

In [ ]:
# ── Deformed shapes at progressive stages ──
ops.wipe()
exec(open("frame_model.py").read())

ops.timeSeries('Constant', 1); ops.pattern('Plain', 100, 1)
for nid, row in node_df.iterrows():
    if row['y'] > 100:
        ops.load(int(nid), 0.0, -W_per_node, 0.0)
ops.constraints('Plain'); ops.numberer('RCM'); ops.system('BandGeneral')
ops.test('NormDispIncr', 1.0e-8, 10); ops.algorithm('Newton')
ops.integrator('LoadControl', 1.0); ops.analysis('Static')
ops.analyze(1); ops.loadConst('-time', 0.0)

ops.timeSeries('Linear', 2); ops.pattern('Plain', 200, 2)
ops.load(roof_node, 1.0, 0.0, 0.0)
ops.test('NormDispIncr', 1.0e-8, 10)
ops.integrator('DisplacementControl', roof_node, 1, d_disp)
ops.analysis('Static')

targets = [n_steps//4, n_steps//2, 3*n_steps//4, len(disp_roof)-2]
shapes = {}
for i in range(max(targets)+1):
    ok = ops.analyze(1)
    if ok != 0:
        ops.algorithm('NewtonLineSearch', 0.8); ok = ops.analyze(1); ops.algorithm('Newton')
    if ok != 0: break
    if i in targets:
        shapes[f"{ops.nodeDisp(roof_node,1):.0f} mm"] = {
            int(nid): (ops.nodeDisp(int(nid),1), ops.nodeDisp(int(nid),2))
            for nid in node_df.index
        }
ops.wipe()

fig, ax = plt.subplots(figsize=(14, 7))
scale = 5.0
for _, row in elem_df.iterrows():
    n1, n2 = row['nodes']
    ax.plot([node_df.loc[n1,'x'], node_df.loc[n2,'x']],
            [node_df.loc[n1,'y'], node_df.loc[n2,'y']], 'k-', lw=1, alpha=0.3)

for (label, shape), color in zip(shapes.items(), ['#2196F3','#FF9800','#F44336','#4CAF50']):
    for _, row in elem_df.iterrows():
        n1, n2 = row['nodes']
        dx1,dy1 = shape.get(n1,(0,0)); dx2,dy2 = shape.get(n2,(0,0))
        ax.plot([node_df.loc[n1,'x']+dx1*scale, node_df.loc[n2,'x']+dx2*scale],
                [node_df.loc[n1,'y']+dy1*scale, node_df.loc[n2,'y']+dy2*scale],
                color=color, lw=2, alpha=0.8)
    ax.plot([],[], color=color, lw=2, label=f'Roof = {label}')

ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
ax.set_title(f'Deformed Shapes ({scale:.0f}x amplified)')
ax.legend(loc='upper right'); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
g.end()if os.path.exists("frame_model.py"):    os.remove("frame_model.py")print("Done.")

---### SummaryThis notebook demonstrated a complete **CAD-to-pushover pipeline**:1. **Fiber sections** — `RectangularColumnSection` meshed three cross-section types2. **CAD import** — IGES from AutoCAD → `removeAllDuplicates()` → physical groups3. **OpenSees bridge** — `g.opensees` translated the mesh into nodes, elements, and BCs4. **Pushover** — Gravity pre-load + displacement-controlled pushover with P-DeltaFor a full nonlinear analysis, replace `elasticBeamColumn` with`forceBeamColumn` and inject the fiber sections via `sec.build(sec_tag=...)`.